# Grand X-Ray Slam: Division A

## コンペ概要
- **タスク**: 胸部X線画像から14種類の胸部疾患を検出する **マルチラベル分類**
- **評価指標**: 14疾患それぞれのAUC-ROCの平均値 (Mean AUC)
- **データ**: 訓練 107,374枚 / テスト 46,233枚

## 14疾患
Atelectasis, Cardiomegaly, Consolidation, Edema, Enlarged Cardiomediastinum,
Fracture, Lung Lesion, Lung Opacity, No Finding, Pleural Effusion,
Pleural Other, Pneumonia, Pneumothorax, Support Devices

## アプローチ
- **モデル**: EfficientNet-B0 (ImageNet事前学習済み) → マルチラベル head
- **損失関数**: BCEWithLogitsLoss (pos_weight でクラス不均衡対応)
- **最適化**: AdamW + CosineAnnealingLR
- **学習高速化**: Mixed Precision (AMP)
- **データ拡張**: RandomHorizontalFlip, RandomRotation, ColorJitter

## 1. セットアップ・インポート

In [ ]:
# ======================================================================
# 1. ライブラリのインポートと基本設定
# ======================================================================
import os
import gc
import time
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as T
import torchvision.models as models

# timm がインストールされていれば使用（Kaggle環境では通常利用可能）
try:
    import timm
    USE_TIMM = True
    print(f"timm version: {timm.__version__}")
except ImportError:
    USE_TIMM = False
    print("timm not available, using torchvision EfficientNet.")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ======================================================================
# 2. ハイパーパラメータ & パス設定
# ======================================================================

class CFG:
    """全設定をまとめるクラス"""
    # --- パス ---
    COMP_DIR = '/kaggle/input/grand-xray-slam-division-a'
    TRAIN_DIR = os.path.join(COMP_DIR, 'train1')
    TEST_DIR = os.path.join(COMP_DIR, 'test1')
    TRAIN_CSV = os.path.join(COMP_DIR, 'train1.csv')
    SAMPLE_SUB = os.path.join(COMP_DIR, 'sample_submission_1.csv')
    OUTPUT_DIR = '/kaggle/working'
    
    # --- モデル ---
    MODEL_NAME = 'tf_efficientnet_b0_ns'  # timm用（NoisyStudent）
    PRETRAINED = True
    NUM_CLASSES = 14
    
    # --- 学習 ---
    IMG_SIZE = 384          # 入力画像サイズ
    BATCH_SIZE = 32         # バッチサイズ（GPU メモリに応じて調整）
    NUM_WORKERS = 2         # DataLoader ワーカー数
    EPOCHS = 5              # エポック数（ベースライン用）
    LR = 1e-4               # 学習率
    WEIGHT_DECAY = 1e-4     # Weight Decay
    SEED = 42
    VAL_RATIO = 0.1         # 検証データの割合
    
    # --- その他 ---
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    USE_AMP = True          # Mixed Precision Training

# 14疾患のカラム名
TARGET_COLS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
    'Lung Opacity', 'No Finding', 'Pleural Effusion',
    'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices'
]

print(f"Device: {CFG.DEVICE}")
print(f"Image size: {CFG.IMG_SIZE}")
print(f"Batch size: {CFG.BATCH_SIZE}")
print(f"Epochs: {CFG.EPOCHS}")

In [ ]:
# ======================================================================
# 3. 再現性のためのシード固定
# ======================================================================

def seed_everything(seed=42):
    """全ての乱数シードを固定して再現性を確保"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.SEED)
print(f"Seed set to {CFG.SEED}")

## 2. データの探索・確認 (EDA)

In [ ]:
# ======================================================================
# 4. データ読み込み & 基本情報の確認
# ======================================================================

# 訓練データの読み込み
train_df = pd.read_csv(CFG.TRAIN_CSV)
print(f"訓練データ shape: {train_df.shape}")
print(f"\nカラム一覧:")
print(train_df.columns.tolist())
print(f"\n先頭5行:")
train_df.head()

In [ ]:
# ======================================================================
# 5. 基本統計量と欠損値の確認
# ======================================================================

print("=" * 60)
print("データ型:")
print(train_df.dtypes)
print("\n" + "=" * 60)
print("欠損値:")
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percent': missing_pct})
print(missing_df[missing_df['Count'] > 0])
print("\n" + "=" * 60)
print("基本統計量:")
train_df.describe()

In [ ]:
# ======================================================================
# 6. ラベル分布の可視化
# ======================================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 各疾患の陽性率
label_counts = train_df[TARGET_COLS].sum().sort_values(ascending=False)
label_rates = (label_counts / len(train_df) * 100)

axes[0].barh(label_rates.index, label_rates.values, color='steelblue')
axes[0].set_xlabel('陽性率 (%)')
axes[0].set_title('各疾患の陽性率')
axes[0].invert_yaxis()
for i, v in enumerate(label_rates.values):
    axes[0].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)

# 1画像あたりのラベル数分布
labels_per_image = train_df[TARGET_COLS].sum(axis=1)
axes[1].hist(labels_per_image, bins=range(0, 15), edgecolor='black',
             color='coral', alpha=0.8, align='left')
axes[1].set_xlabel('1画像あたりのラベル数')
axes[1].set_ylabel('画像数')
axes[1].set_title('1画像あたりのラベル数分布')

plt.tight_layout()
plt.show()

print(f"\n各疾患の陽性数:")
print(label_counts)

In [ ]:
# ======================================================================
# 7. 疾患間の共起関係（相関行列）
# ======================================================================

fig, ax = plt.subplots(figsize=(12, 10))
corr = train_df[TARGET_COLS].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, ax=ax, linewidths=0.5)
ax.set_title('疾患間の相関行列（共起関係）')
plt.tight_layout()
plt.show()

In [ ]:
# ======================================================================
# 8. メタ情報の確認 (Sex, Age, ViewPosition)
# ======================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 性別分布
if 'Sex' in train_df.columns:
    train_df['Sex'].value_counts(dropna=False).plot.bar(ax=axes[0], color='steelblue')
    axes[0].set_title('性別分布')
    axes[0].set_ylabel('画像数')

# 年齢分布
if 'Age' in train_df.columns:
    train_df['Age'].dropna().hist(bins=50, ax=axes[1], color='coral', edgecolor='black')
    axes[1].set_title('年齢分布')
    axes[1].set_xlabel('年齢')

# ViewPosition分布
if 'ViewPosition' in train_df.columns:
    train_df['ViewPosition'].value_counts(dropna=False).plot.bar(
        ax=axes[2], color='seagreen')
    axes[2].set_title('撮影位置 (ViewPosition) 分布')

plt.tight_layout()
plt.show()

In [ ]:
# ======================================================================
# 9. サンプル画像の表示
# ======================================================================

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
sample_rows = train_df.sample(8, random_state=CFG.SEED)

for idx, (ax, (_, row)) in enumerate(zip(axes.flat, sample_rows.iterrows())):
    img_path = os.path.join(CFG.TRAIN_DIR, row['Image_Name'])
    try:
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img, cmap='gray')
        # 陽性ラベルを取得
        positive_labels = [col for col in TARGET_COLS if row[col] == 1]
        title = ', '.join(positive_labels[:2])  # 最初の2つだけ表示
        if len(positive_labels) > 2:
            title += f' (+{len(positive_labels)-2})'
        ax.set_title(title, fontsize=8)
    except Exception as e:
        ax.set_title(f'Error: {e}', fontsize=8)
    ax.axis('off')

plt.suptitle('訓練データ サンプル画像', fontsize=14)
plt.tight_layout()
plt.show()

## 3. 前処理・データセット構築

In [ ]:
# ======================================================================
# 10. サンプル提出ファイルの確認
# ======================================================================

sample_sub = pd.read_csv(CFG.SAMPLE_SUB)
print(f"提出ファイル shape: {sample_sub.shape}")
print(f"カラム: {sample_sub.columns.tolist()}")
sample_sub.head()

In [ ]:
# ======================================================================
# 11. 訓練/検証データの分割
#     - Patient_IDベースで分割（患者のリークを防ぐ）
# ======================================================================

# Patient_ID が存在する場合はそれを使い、
# 同一患者の画像が train/val に分かれないようにする
if 'Patient_ID' in train_df.columns:
    patient_ids = train_df['Patient_ID'].unique()
    train_patients, val_patients = train_test_split(
        patient_ids, test_size=CFG.VAL_RATIO, random_state=CFG.SEED
    )
    train_data = train_df[train_df['Patient_ID'].isin(train_patients)].reset_index(drop=True)
    val_data = train_df[train_df['Patient_ID'].isin(val_patients)].reset_index(drop=True)
else:
    # Patient_IDがない場合はランダム分割
    train_data, val_data = train_test_split(
        train_df, test_size=CFG.VAL_RATIO, random_state=CFG.SEED
    )
    train_data = train_data.reset_index(drop=True)
    val_data = val_data.reset_index(drop=True)

print(f"訓練データ: {len(train_data)} 枚")
print(f"検証データ: {len(val_data)} 枚")

# 訓練データの各ラベル陽性数からpos_weightを計算
pos_counts = train_data[TARGET_COLS].sum().values
neg_counts = len(train_data) - pos_counts
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float32)
print(f"\npos_weight (クラス不均衡対応):")
for col, w in zip(TARGET_COLS, pos_weight):
    print(f"  {col}: {w:.2f}")

In [ ]:
# ======================================================================
# 12. データ拡張 (Transforms) の定義
# ======================================================================

# ImageNet の平均・標準偏差（事前学習済みモデル用）
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def get_transforms(phase='train'):
    """学習/検証/テスト用の画像変換を返す"""
    if phase == 'train':
        return T.Compose([
            T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=10),
            T.ColorJitter(brightness=0.2, contrast=0.2),
            T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:
        return T.Compose([
            T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

print("Transforms 定義完了")
print(f"Train transforms: {get_transforms('train')}")

In [ ]:
# ======================================================================
# 13. PyTorch Dataset クラスの定義
# ======================================================================

class ChestXrayDataset(Dataset):
    """胸部X線画像のマルチラベル分類用データセット"""
    
    def __init__(self, df, img_dir, target_cols, transform=None, is_test=False):
        """
        Args:
            df: DataFrame（Image_Name と各ラベルカラムを含む）
            img_dir: 画像ディレクトリのパス
            target_cols: ターゲットカラム名のリスト
            transform: 画像変換
            is_test: テストデータかどうか
        """
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.target_cols = target_cols
        self.transform = transform
        self.is_test = is_test
        
        # Image_Name カラムの名前を自動検出
        img_col_candidates = ['Image_Name', 'Image_name', 'image_name']
        self.img_col = None
        for col in img_col_candidates:
            if col in self.df.columns:
                self.img_col = col
                break
        if self.img_col is None:
            # 最初のカラムをImage列として推定
            self.img_col = self.df.columns[0]
            print(f"Warning: Image column not found, using '{self.img_col}'")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row[self.img_col]
        img_path = os.path.join(self.img_dir, img_name)
        
        # 画像の読み込み（エラー時は黒画像で代替）
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Warning: {img_path} の読み込みに失敗: {e}")
            image = Image.new('RGB', (CFG.IMG_SIZE, CFG.IMG_SIZE), (0, 0, 0))
        
        if self.transform:
            image = self.transform(image)
        
        if self.is_test:
            return image
        
        # ラベルの取得
        labels = row[self.target_cols].values.astype(np.float32)
        labels = torch.tensor(labels, dtype=torch.float32)
        
        return image, labels

print("Dataset クラス定義完了")

In [ ]:
# ======================================================================
# 14. DataLoader の作成
# ======================================================================

train_dataset = ChestXrayDataset(
    df=train_data, img_dir=CFG.TRAIN_DIR, target_cols=TARGET_COLS,
    transform=get_transforms('train'), is_test=False
)
val_dataset = ChestXrayDataset(
    df=val_data, img_dir=CFG.TRAIN_DIR, target_cols=TARGET_COLS,
    transform=get_transforms('val'), is_test=False
)

train_loader = DataLoader(
    train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True,
    num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
    num_workers=CFG.NUM_WORKERS, pin_memory=True
)

# 動作確認
sample_batch = next(iter(train_loader))
print(f"バッチ画像 shape: {sample_batch[0].shape}")
print(f"バッチラベル shape: {sample_batch[1].shape}")
print(f"ラベル例: {sample_batch[1][0]}")

## 4. モデル構築

In [ ]:
# ======================================================================
# 15. モデル定義：EfficientNet-B0 ベース
# ======================================================================

class ChestXrayModel(nn.Module):
    """EfficientNet-B0 ベースのマルチラベル分類モデル"""
    
    def __init__(self, model_name=CFG.MODEL_NAME, num_classes=CFG.NUM_CLASSES,
                 pretrained=CFG.PRETRAINED):
        super().__init__()
        
        if USE_TIMM:
            # timm を使用（より多くの事前学習済みモデルが利用可能）
            self.backbone = timm.create_model(
                model_name, pretrained=pretrained, num_classes=0
            )
            in_features = self.backbone.num_features
        else:
            # torchvision の EfficientNet-B0
            weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
            self.backbone = models.efficientnet_b0(weights=weights)
            in_features = self.backbone.classifier[1].in_features
            self.backbone.classifier = nn.Identity()  # 分類頭を除去
        
        # カスタム分類ヘッド
        self.head = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(p=0.3),
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )
        
        print(f"Model: {model_name if USE_TIMM else 'efficientnet_b0'}")
        print(f"Backbone features: {in_features}")
        print(f"Output classes: {num_classes}")
    
    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

# モデル作成
model = ChestXrayModel().to(CFG.DEVICE)

# パラメータ数の確認
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n総パラメータ数: {total_params:,}")
print(f"学習可能パラメータ数: {trainable_params:,}")

## 5. 学習ループ

In [ ]:
# ======================================================================
# 16. 損失関数・オプティマイザ・スケジューラ
# ======================================================================

# BCEWithLogitsLoss（pos_weight でクラス不均衡対応）
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight.to(CFG.DEVICE)
)

# AdamW オプティマイザ
optimizer = optim.AdamW(
    model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY
)

# CosineAnnealing スケジューラ
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG.EPOCHS, eta_min=1e-6
)

# Mixed Precision 用 GradScaler
scaler = GradScaler(enabled=CFG.USE_AMP)

print("損失関数: BCEWithLogitsLoss (with pos_weight)")
print(f"オプティマイザ: AdamW (lr={CFG.LR}, wd={CFG.WEIGHT_DECAY})")
print(f"スケジューラ: CosineAnnealingLR (T_max={CFG.EPOCHS})")
print(f"Mixed Precision: {CFG.USE_AMP}")

In [ ]:
# ======================================================================
# 17. 学習・検証関数の定義
# ======================================================================

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    """1エポック分の学習を実行"""
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(loader, desc='Training', leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        with autocast(enabled=CFG.USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss


def validate(model, loader, criterion, device):
    """検証を実行し、損失とAUCを返す"""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validation', leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            with autocast(enabled=CFG.USE_AMP):
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            
            # sigmoid で確率に変換
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_preds.append(probs)
            all_labels.append(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(loader.dataset)
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    
    # 各疾患のAUCを計算
    aucs = []
    for i, col in enumerate(TARGET_COLS):
        try:
            auc = roc_auc_score(all_labels[:, i], all_preds[:, i])
        except ValueError:
            # ラベルが1種類しかない場合
            auc = 0.5
        aucs.append(auc)
    
    mean_auc = np.mean(aucs)
    
    return epoch_loss, mean_auc, aucs

print("学習・検証関数定義完了")

In [ ]:
# ======================================================================
# 18. 学習実行
# ======================================================================

best_auc = 0.0
best_epoch = 0
history = {'train_loss': [], 'val_loss': [], 'val_auc': []}

print("=" * 70)
print(f"学習開始: {CFG.EPOCHS} エポック")
print(f"訓練データ: {len(train_data)} 枚, 検証データ: {len(val_data)} 枚")
print("=" * 70)

for epoch in range(CFG.EPOCHS):
    start_time = time.time()
    
    # --- 学習 ---
    train_loss = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, CFG.DEVICE
    )
    
    # --- 検証 ---
    val_loss, val_auc, val_aucs = validate(
        model, val_loader, criterion, CFG.DEVICE
    )
    
    # スケジューラ更新
    scheduler.step()
    
    # 履歴保存
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    
    elapsed = time.time() - start_time
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"\nEpoch {epoch+1}/{CFG.EPOCHS} ({elapsed:.0f}s)")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  Val AUC:    {val_auc:.4f} (Mean)")
    print(f"  LR:         {current_lr:.6f}")
    
    # 各疾患のAUCを表示
    print("  Per-class AUC:")
    for col, auc in zip(TARGET_COLS, val_aucs):
        print(f"    {col:30s}: {auc:.4f}")
    
    # ベストモデルを保存
    if val_auc > best_auc:
        best_auc = val_auc
        best_epoch = epoch + 1
        torch.save(model.state_dict(),
                   os.path.join(CFG.OUTPUT_DIR, 'best_model.pth'))
        print(f"  >>> Best model saved! (AUC: {best_auc:.4f})")

print("\n" + "=" * 70)
print(f"学習完了! Best Epoch: {best_epoch}, Best Mean AUC: {best_auc:.4f}")
print("=" * 70)

## 6. 学習結果の可視化

In [ ]:
# ======================================================================
# 19. 学習曲線の可視化
# ======================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(range(1, CFG.EPOCHS+1), history['train_loss'], 'b-o', label='Train Loss')
axes[0].plot(range(1, CFG.EPOCHS+1), history['val_loss'], 'r-o', label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('学習曲線 (Loss)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# AUC
axes[1].plot(range(1, CFG.EPOCHS+1), history['val_auc'], 'g-o', label='Val Mean AUC')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean AUC')
axes[1].set_title('検証 Mean AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=best_auc, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_auc:.4f}')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. テストデータの予測 & 提出ファイル生成

In [ ]:
# ======================================================================
# 20. ベストモデルのロード
# ======================================================================

# ベストモデルをロード
model.load_state_dict(
    torch.load(os.path.join(CFG.OUTPUT_DIR, 'best_model.pth'),
               map_location=CFG.DEVICE)
)
model.eval()
print(f"ベストモデル (Epoch {best_epoch}, AUC {best_auc:.4f}) をロードしました")

In [ ]:
# ======================================================================
# 21. テストデータの推論
# ======================================================================

# テストデータの準備
test_dataset = ChestXrayDataset(
    df=sample_sub, img_dir=CFG.TEST_DIR, target_cols=TARGET_COLS,
    transform=get_transforms('test'), is_test=True
)
test_loader = DataLoader(
    test_dataset, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
    num_workers=CFG.NUM_WORKERS, pin_memory=True
)

print(f"テストデータ: {len(test_dataset)} 枚")
print(f"バッチ数: {len(test_loader)}")

# 推論実行
all_preds = []

with torch.no_grad():
    for images in tqdm(test_loader, desc='Test Inference'):
        images = images.to(CFG.DEVICE, non_blocking=True)
        
        with autocast(enabled=CFG.USE_AMP):
            outputs = model(images)
        
        probs = torch.sigmoid(outputs).cpu().numpy()
        all_preds.append(probs)

all_preds = np.concatenate(all_preds, axis=0)
print(f"\n予測結果 shape: {all_preds.shape}")
print(f"予測確率の統計:")
print(f"  Min: {all_preds.min():.4f}")
print(f"  Max: {all_preds.max():.4f}")
print(f"  Mean: {all_preds.mean():.4f}")

In [ ]:
# ======================================================================
# 22. TTA (Test-Time Augmentation) - オプション
#     水平反転での推論結果を平均して精度を向上
# ======================================================================

print("TTA (水平反転) を実行中...")

tta_transform = T.Compose([
    T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    T.RandomHorizontalFlip(p=1.0),  # 必ず反転
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

tta_dataset = ChestXrayDataset(
    df=sample_sub, img_dir=CFG.TEST_DIR, target_cols=TARGET_COLS,
    transform=tta_transform, is_test=True
)
tta_loader = DataLoader(
    tta_dataset, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
    num_workers=CFG.NUM_WORKERS, pin_memory=True
)

tta_preds = []
with torch.no_grad():
    for images in tqdm(tta_loader, desc='TTA Inference'):
        images = images.to(CFG.DEVICE, non_blocking=True)
        with autocast(enabled=CFG.USE_AMP):
            outputs = model(images)
        probs = torch.sigmoid(outputs).cpu().numpy()
        tta_preds.append(probs)

tta_preds = np.concatenate(tta_preds, axis=0)

# 元の予測とTTAの平均
final_preds = (all_preds + tta_preds) / 2.0
print(f"TTA後の予測 shape: {final_preds.shape}")
print(f"予測確率の統計 (TTA後):")
print(f"  Min: {final_preds.min():.4f}, Max: {final_preds.max():.4f}, Mean: {final_preds.mean():.4f}")

In [ ]:
# ======================================================================
# 23. 提出ファイルの作成
# ======================================================================

# サンプル提出ファイルのImage_Nameカラム名を検出
img_col = None
for col in ['Image_Name', 'Image_name', 'image_name']:
    if col in sample_sub.columns:
        img_col = col
        break
if img_col is None:
    img_col = sample_sub.columns[0]

# 提出DataFrameの作成
submission = pd.DataFrame()
submission[img_col] = sample_sub[img_col]

for i, col in enumerate(TARGET_COLS):
    submission[col] = final_preds[:, i]

# 確率値を0-1にクリップ（念のため）
for col in TARGET_COLS:
    submission[col] = submission[col].clip(0.0, 1.0)

# CSVファイルとして保存
submission_path = os.path.join(CFG.OUTPUT_DIR, 'submission.csv')
submission.to_csv(submission_path, index=False)

print(f"提出ファイル保存完了: {submission_path}")
print(f"Shape: {submission.shape}")
print(f"\n先頭5行:")
submission.head()

In [ ]:
# ======================================================================
# 24. 提出ファイルの検証
# ======================================================================

print("=" * 60)
print("提出ファイルの検証")
print("=" * 60)

# Shape の確認
expected_rows = len(sample_sub)
expected_cols = 1 + CFG.NUM_CLASSES  # Image_Name + 14 labels
assert submission.shape == (expected_rows, expected_cols), \
    f"Shape mismatch: {submission.shape} != ({expected_rows}, {expected_cols})"
print(f"✓ Shape OK: {submission.shape}")

# カラム名の確認
expected_label_cols = set(TARGET_COLS)
actual_label_cols = set(submission.columns) - {img_col}
assert expected_label_cols == actual_label_cols, \
    f"Column mismatch: {expected_label_cols.symmetric_difference(actual_label_cols)}"
print(f"✓ カラム名 OK")

# 値の範囲確認
for col in TARGET_COLS:
    assert submission[col].between(0, 1).all(), f"{col} has values outside [0, 1]"
print(f"✓ 全ての予測値が [0, 1] の範囲内")

# 欠損値の確認
assert submission.isnull().sum().sum() == 0, "NaN values found"
print(f"✓ 欠損値なし")

# Image_Name の一致確認
assert list(submission[img_col]) == list(sample_sub[img_col]), "Image names mismatch"
print(f"✓ Image_Name が sample_submission と一致")

print(f"\n各ラベルの予測統計:")
print(submission[TARGET_COLS].describe().round(4))

print("\n" + "=" * 60)
print("全ての検証をパスしました！提出準備完了です。")
print("=" * 60)

In [ ]:
# ======================================================================
# 25. 予測分布の可視化
# ======================================================================

fig, axes = plt.subplots(3, 5, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(TARGET_COLS):
    axes[i].hist(submission[col], bins=50, color='steelblue',
                 edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('予測確率')
    axes[i].set_xlim(0, 1)

# 余分なサブプロットを非表示
for i in range(len(TARGET_COLS), len(axes)):
    axes[i].axis('off')

plt.suptitle('各疾患の予測確率分布', fontsize=14)
plt.tight_layout()
plt.show()

## 8. まとめ & 改善のヒント

### 実行結果
- **モデル**: EfficientNet-B0 (NoisyStudent事前学習)
- **検証 Mean AUC**: 上記の学習ログを参照
- **提出ファイル**: `submission.csv` (46,233行 × 15列)

### 改善のためのアイデア

| カテゴリ | アイデア | 期待効果 |
|---------|---------|--------|
| **モデル** | EfficientNet-B2/B4/B7 に変更 | 精度↑ |
| **モデル** | DenseNet-121 (CheXpertベースライン) | 医療画像での実績 |
| **モデル** | Vision Transformer (ViT) | 大規模データに有効 |
| **アンサンブル** | 複数モデルの平均 | 安定性↑ |
| **データ拡張** | Albumentations (GridDistortion等) | 汎化性能↑ |
| **学習戦略** | StratifiedKFold交差検証 | 信頼性↑ |
| **学習戦略** | Progressive Resizing (224→384→512) | 精度↑ |
| **損失関数** | Focal Loss (希少疾患対応) | 不均衡↑ |
| **後処理** | 疾患間の共起関係を利用した補正 | 精度微↑ |
| **メタ情報** | Age/Sex/ViewPosition を追加特徴として利用 | 精度↑ |

In [ ]:
# ======================================================================
# 26. メモリ解放
# ======================================================================

del model, train_loader, val_loader, test_loader
del train_dataset, val_dataset, test_dataset
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("メモリ解放完了")
print(f"\n出力ファイル:")
for f in os.listdir(CFG.OUTPUT_DIR):
    fpath = os.path.join(CFG.OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024**2)
        print(f"  {f}: {size_mb:.2f} MB")